# 02. Literary & Video Game Localization Pipelines

**Preset:** `TranslationPreset`

## Why `canonarc` Fits This Domain

Sequential LLM translation of a 300-page novel or a 500,000-word JRPG fails in four predictable ways when using standard sliding windows or vanilla RAG:

1. **Grammatical & Exonym Drift:** A character's grammatical gender, declension class, or localized target name (`target_name`) shifts mid-book.
2. **T–V Register Collapse:** Languages with formal/informal address (Serbian/Croatian *Vi*/*ti*, French *vous*/*tu*, German *Sie*/*du*, Japanese/Korean honorifics) require **asymmetric directed registers**—character A may address B formally (*Vi*) while B addresses A informally (*ti*), and that register may flip permanently after a pivotal scene.
3. **Voice Flattening:** Without a gated voice bank, a character's distinct syntax regresses toward generic "translationese."
4. **Flashbacks Corrupting Present State:** Translating a flashback chapter shouldn't overwrite the characters' present-day relationship registers.

In this notebook, we build a chunk-by-chunk localization pipeline using `TranslationPreset`, asymmetric dyads, multi-party voice anchoring, and flashback-safe ingestion (`is_flashback=True`).

In [1]:
from canonarc import (
    EdgeDelta,
    EntityDelta,
    EntityRecord,
    EpisodeRecord,
    CanonArcEngine,
    StepDelta,
    TranslationPreset,
    VoiceExemplar,
)

engine = CanonArcEngine(":memory:", preset=TranslationPreset())
print("Localization engine ready with TranslationPreset.")

Localization engine ready with TranslationPreset.


## Step 1: Lock Tier 1 Grammatical Metadata & Asymmetric Tier 2 Registers

We seed three characters for an English → Serbian/Croatian literary localization pipeline:
- `base_attributes` locks `target_name`, `gender`, and `declension` so every prompt enforces exact morphology.
- Initial edges establish an **asymmetric hierarchy**:
  - Captain Elena → Lady Astrid: `Vi` (formal, deferential)
  - Lady Astrid → Captain Elena: `ti` (informal, warm)
  - Captain Elena → Scout Marko: `ti` (informal, brisk)

In [2]:
engine.register_entity(
    EntityRecord(
        entity_id="char_elena",
        canonical_name="Elena",
        base_attributes={"target_name": "Jelena", "gender": "F", "declension": "a-decl"},
        state={"mood": "composed"},
    ),
    aliases=["Captain Elena", "the Captain"],
    step=0,
)
engine.register_entity(
    EntityRecord(
        entity_id="char_astrid",
        canonical_name="Astrid",
        base_attributes={"target_name": "Astrid", "gender": "F", "declension": "indecl"},
        state={"mood": "serene"},
    ),
    aliases=["Lady Astrid", "the Baroness"],
    step=0,
)
engine.register_entity(
    EntityRecord(
        entity_id="char_marko",
        canonical_name="Marko",
        base_attributes={"target_name": "Marko", "gender": "M", "declension": "o-decl"},
        state={"mood": "alert"},
    ),
    aliases=["the young scout"],
    step=0,
)

# Asymmetric initial registers at Chunk 1
engine.commit_step(
    step=1,
    delta=StepDelta(
        edges=[
            EdgeDelta(source_id="char_elena", target_id="char_astrid", state={"register": "Vi", "tone": "deferential"}),
            EdgeDelta(source_id="char_astrid", target_id="char_elena", state={"register": "ti", "tone": "warm"}),
            EdgeDelta(source_id="char_elena", target_id="char_marko", state={"register": "ti", "tone": "brisk"}),
        ]
    ),
)

# Index high-quality translated exemplars into Tier 5 Voice Bank
engine.index_voice_exemplars(
    step=5,
    exemplars=[
        VoiceExemplar(
            step=5,
            speaker_id="char_elena",
            addressee_id="char_astrid",
            text="Molim Vas da ostanete iza zidina dok izvidnica ne proveri severni prevoj, gospođo.",
            source_text="Please remain behind the walls until the scouts check the northern pass, my lady.",
            metadata={"register": "Vi", "tone": "deferential"},
            quality_score=9.5,
        ),
        VoiceExemplar(
            step=5,
            speaker_id="char_elena",
            addressee_id="char_marko",
            text="Uzmi konja i proveri severni prevoj pre nego što padne magla, ne čekaj zoru.",
            source_text="Take a horse and check the northern pass before fog falls; don't wait for dawn.",
            metadata={"register": "ti", "tone": "brisk"},
            quality_score=9.3,
        ),
    ],
)
print("Locked Tier 1 grammar, asymmetric T-V registers, and initial voice anchors.")

Locked Tier 1 grammar, asymmetric T-V registers, and initial voice anchors.


## Step 2: Multi-Party Scene Retrieval (Same Speaker, Two Different Registers)

In Chunk 12, Captain Elena addresses **both** Lady Astrid (formal `Vi`) and the young scout Marko (informal `ti`) in the same scene.

Because `VoiceBank` resolves edges per dyad (`source -> target`), `TranslationPreset` surfaces the exact binding register for each pair and retrieves matching stylistic exemplars.

In [3]:
chunk_12_source = (
    'Captain Elena bowed to the Baroness, then turned sharply to the young scout. '
    '"My lady, the pass is clear. You—saddle the horses at once."'
)
ctx_12 = engine.retrieve_context(step=12, text=chunk_12_source, top_k_voice=4)
print(engine.format_context(ctx_12))

## Scene character state & register — binding
- Astrid (char_astrid): target='Astrid', F, indecl [mood: serene]
- Elena (char_elena): target='Jelena', F, a-decl [mood: composed]
- Marko (char_marko): target='Marko', M, o-decl [mood: alert]
- char_astrid → char_elena: register=ti, tone=warm
- char_elena → char_astrid: register=Vi, tone=deferential
- char_elena → char_marko: register=ti, tone=brisk

## Character voice bank — stylistic anchor
- char_elena (Vi): "Molim Vas da ostanete iza zidina dok izvidnica ne proveri severni prevoj, gospođo."
- char_elena (ti): "Uzmi konja i proveri severni prevoj pre nego što padne magla, ne čekaj zoru."


## Step 3: Dynamic Register Shift After a Pivotal Plot Event (Chunk 40)

At Chunk 40, Elena pulls Astrid from a collapsing bridge; Astrid insists they drop courtly formality (`Vi` → `ti`). We commit the shift inside `step_transaction()` alongside a new informal exemplar for Elena speaking to Astrid.

In [4]:
with engine.step_transaction(step=40, mentions=["Elena", "Astrid"]) as tx:
    tx.commit(
        delta=StepDelta(
            edges=[
                EdgeDelta(
                    source_id="char_elena",
                    target_id="char_astrid",
                    state={"register": "ti", "tone": "devoted"},
                    caused_by_episode_id="ep_chunk40_bridge_rescue",
                    reason="Astrid asked Elena to drop formal Vi after the bridge rescue",
                )
            ],
            episodes=[
                EpisodeRecord(
                    episode_id="ep_chunk40_bridge_rescue",
                    predicate="rescue",
                    summary="Elena pulled Astrid from the collapsing stone bridge; they swore informal camaraderie.",
                    participants={"char_elena": "rescuer", "char_astrid": "rescued"},
                    witnesses=["char_elena", "char_astrid"],
                    salience=9.2,
                )
            ],
        ),
        exemplars=[
            VoiceExemplar(
                step=40,
                speaker_id="char_elena",
                addressee_id="char_astrid",
                text="Drži se čvrsto za mene, Astrid—neću te pustiti dok ne stignemo do druge obale!",
                source_text="Hold tight to me, Astrid—I won't let go until we reach the other bank!",
                metadata={"register": "ti", "tone": "devoted"},
                quality_score=9.4,
            )
        ],
    )

# Retrieve Chunk 45: Elena -> Astrid is now binding 'ti' and pulls the new 'ti' exemplar!
ctx_45 = engine.retrieve_context(step=45, text="Elena handed Astrid a warm cloak by the fire.")
print(engine.format_context(ctx_45))

## Scene character state & register — binding
- Astrid (char_astrid): target='Astrid', F, indecl [mood: serene]
- Elena (char_elena): target='Jelena', F, a-decl [mood: composed]
- char_astrid → char_elena: register=ti, tone=warm
- char_elena → char_astrid: register=ti, tone=devoted

## Relevant narrative episodes — context
- [Step 40 | known to: char_astrid, char_elena] Elena pulled Astrid from the collapsing stone bridge; they swore informal camaraderie.

## Character voice bank — stylistic anchor
- char_elena (ti): "Drži se čvrsto za mene, Astrid—neću te pustiti dok ne stignemo do druge obale!"
- char_elena (ti): "Uzmi konja i proveri severni prevoj pre nego što padne magla, ne čekaj zoru."


## Step 4: Flashback-Safe Ingestion (`is_flashback=True`)

At Chunk 46, the novel inserts a **flashback** to five years earlier, when Elena was a nervous recruit. Setting `StepDelta(is_flashback=True, story_time="5_years_ago")` indexes the flashback episode into Tier 6 **without overwriting** Elena's present-day `composed` state or her `ti` register with Astrid.

In [5]:
engine.commit_step(
    step=46,
    delta=StepDelta(
        is_flashback=True,
        story_time="5_years_ago",
        entities=[
            # Ignored for existing entities because is_flashback=True!
            EntityDelta(entity_id="char_elena", state={"mood": "terrified_recruit"}),
        ],
        episodes=[
            EpisodeRecord(
                episode_id="ep_chunk46_academy_flashback",
                predicate="flashback",
                summary="Five years earlier, Elena failed her first winter crossing at the northern academy.",
                participants={"char_elena": "subject"},
                witnesses=["char_elena"],
                story_time="5_years_ago",
                salience=7.0,
            )
        ],
    ),
)

# Confirm Elena's present-day state was NOT corrupted by the flashback
ctx_47 = engine.retrieve_context(step=47, mentions=["Elena", "Astrid"])
elena = [e for e in ctx_47.entities if e.entity_id == "char_elena"][0]
elena_to_astrid = [e for e in ctx_47.active_edges if e.source_id == "char_elena"][0]

print("Present-day Elena mood (preserved):", elena.state["mood"])
print("Present-day Elena->Astrid register (preserved):", elena_to_astrid.state["register"])
print("Flashback indexed in Tier 6:", [ep.episode_id for ep in ctx_47.episodes])

assert elena.state["mood"] == "composed"
assert elena_to_astrid.state["register"] == "ti"
assert any(ep.episode_id == "ep_chunk46_academy_flashback" for ep in ctx_47.episodes)

engine.close()

Present-day Elena mood (preserved): composed
Present-day Elena->Astrid register (preserved): ti
Flashback indexed in Tier 6: ['ep_chunk40_bridge_rescue', 'ep_chunk46_academy_flashback']
